#  Análisis Táctico, Perfilamiento SOM y Scouting (Moneyball)

## 1. Importación de la Base de Datos Clusterizada

En este script nos desvinculamos del entrenamiento geométrico de la red neuronal. Nuestro único objetivo es leer la base de datos resultante del **Script 5** y traducir las agrupaciones matemáticas (Arquetipos SOM) al idioma real del fútbol.

El código está diseñado para ser **dinámico**: detectará automáticamente cuántos clústeres se generaron en el paso anterior, permitiéndonos cambiar la configuración en el futuro sin tener que reescribir este cuaderno.

In [9]:
import pandas as pd
import numpy as np
from scipy.stats import norm

print("="*85)
print(" 🔄 INICIANDO SCRIPT 5: SCOUTING Y ANÁLISIS DE ARQUETIPOS")
print("="*85)

# 1. Cargar el Súper Dataset Definitivo (Ya trae crudos, precios, Z-Scores, PCA y Neuronas)
ruta_clusters = '/content/drive/MyDrive/dataset_final_clusters.csv'
df_clusters = pd.read_csv(ruta_clusters)

# 2. Calculamos dinámicamente el total de macro-clústeres para los reportes
total_clusters = df_clusters['Arquetipo_SOM'].nunique()

print(f"✅ ¡Base maestra cargada exitosamente!")
print(f"✅ Se detectaron automáticamente {total_clusters} Arquetipos Tácticos (Macro-Clústeres).")
print(f"✅ El dataset tiene {df_clusters.shape[1]} columnas (No se necesitan cruces adicionales).")
print("-" * 85)

# Vista previa para confirmar que tenemos todo en un solo lugar
display(df_clusters[['player', 'value_eur', 'xG P90', 'xG P90_z', 'PC1', 'ID_Neurona', 'Arquetipo_SOM']].head())

 🔄 INICIANDO SCRIPT 5: SCOUTING Y ANÁLISIS DE ARQUETIPOS
✅ ¡Base maestra cargada exitosamente!
✅ Se detectaron automáticamente 23 Arquetipos Tácticos (Macro-Clústeres).
✅ El dataset tiene 52 columnas (No se necesitan cruces adicionales).
-------------------------------------------------------------------------------------


,player,value_eur,xG P90,xG P90_z,PC1,ID_Neurona,Arquetipo_SOM
0,Aaron Cresswell,3000000.0,0.021197,-0.752355,1.226360,8-17,12
1,Aaron Lennon,625000.0,0.082510,0.452495,0.899998,5-7,17
2,Aaron Ramsey,26000000.0,0.234788,1.627059,5.103720,12-14,18
3,Abdelaziz Barrada,4300000.0,0.038600,-0.333183,-2.349870,0-5,11
4,Abdelaziz Barrada,4300000.0,0.133185,1.030335,2.253579,8-13,10


## 2. Radiografía de los Arquetipos (Z-Score y Percentiles)

Para entender qué rol táctico juega cada grupo en la cancha, agruparemos a los jugadores por su `Arquetipo_SOM` y calcularemos el promedio de sus métricas.

Dado que nuestros datos están estandarizados, utilizamos una doble lectura para máxima claridad:
1. **Z-Score (El Valor Crudo):** Indica a cuántas desviaciones estándar está el clúster respecto al jugador europeo promedio (0.00).
2. **Percentiles (La Traducción Visual):** Usando la Función de Distribución Acumulada Gaussiana (CDF), traducimos el Z-Score a un rango del 0 al 100. Un percentil 90 indica que ese grupo es superior al 90% de la liga en esa métrica.

Además, rankearemos los clústeres desde el que tiene mayor **Impacto Global** (los "todoterreno" o estrellas absolutas) hasta los de menor impacto (jugadores de rol muy específico o defensores rígidos).

---

### 🧠 El Éxito del Mapa Autoorganizado (SOM)

El mayor logro de esta red neuronal es que **destruyó las etiquetas tradicionales de posición**. En lugar de agrupar "delanteros con delanteros" o "defensas con defensas", el SOM entendió la complejidad del fútbol moderno: agrupó a los jugadores estrictamente por su **función táctica y comportamiento geométrico** en la cancha.

### 🌟 Radiografía de los Arquetipos de Élite

La diferenciación en el Top 4 demuestra cómo el algoritmo logró separar perfiles complejos con precisión quirúrgica:

* **Arquetipo 18 (Los Creadores Totales - #1):** Es la élite ofensiva absoluta. Agrupa a extremos y mediapuntas (Neymar, David Silva, Dybala) que dominan el último tercio. Tienen números de ataque, retención y creación estratosféricos (Percentil 90+ en asistencias, conducciones y faltas recibidas). Son el motor de peligro de sus equipos.
* **Arquetipo 3 (Los Metrónomos y Carrileros - #2):** Una genialidad matemática. El SOM agrupó a laterales ofensivos (Marcelo, Filipe Luís) con pivotes organizadores (Modrić, Fàbregas). Geométricamente, ambos roles hacen exactamente lo mismo: recuperar el balón, progresar líneas y mantener la estructura del equipo conectada (Percentil 90+ en centralidad, intercepciones y pases progresivos).
* **Arquetipo 10 (Los Atacantes Dinámicos - #3):** Perfiles como Antoine Griezmann o Lucas Moura. Son jugadores de alta movilidad que no solo definen, sino que conducen el balón largas distancias, rompen líneas y ayudan en la recuperación alta.
* **Arquetipo 14 (Los Depredadores de Área - #4):** Zlatan Ibrahimović, Romelu Lukaku. El algoritmo identificó a los finalizadores puros. Tienen un impacto brutal en la métrica de goles esperados (xG en Percentil 92), pero nula participación defensiva y baja creación. Viven de espaldas al arco y para definir.

### 🛡️ La Inteligencia de Aislación

La prueba definitiva de que el modelo funciona sin sesgos humanos está en el **Arquetipo 5 (#22)**. Sin que le entregáramos ninguna regla sobre posiciones nominales, el algoritmo agrupó a 103 porteros y los aisló en su propia neurona geométrica. Descubrió empíricamente la posición del arquero al detectar un patrón de ataque nulo, intercepciones nulas, pero un volumen atípico de pases largos progresivos.

In [10]:
import pandas as pd
import numpy as np
from scipy.stats import norm

# ==============================================================================
# 0. FUNCIÓN DE ARQUITECTURA DE POSICIONES (CORREGIDA)
# ==============================================================================
def clasificar_macro_posicion(posicion):
    pos = str(posicion).upper()

    # 1. MEDIOCAMPISTAS PRIMERO: Atrapa "Defensive Midfield", "Central Midfield", etc.
    # antes de que las palabras "Defensive" o "Central" los manden a otras líneas.
    if any(x in pos for x in ['MF', 'MID', 'MIDFIELDER', 'CM', 'AM', 'DM', 'RM', 'LM', 'VOLANTE', 'PIVOTE', 'MEDIO']):
        return '🧠 MEDIOCAMPISTAS'

    # 2. DELANTEROS
    elif any(x in pos for x in ['FW', 'DEL', 'ST', 'CF', 'LW', 'RW', 'EXTREMO', 'PUNTA', 'ATT', 'FORWARD']):
        return '🎯 DELANTEROS'

    # 3. DEFENSAS
    elif any(x in pos for x in ['DF', 'DEF', 'CB', 'LB', 'RB', 'WB', 'LATERAL', 'CENTRAL', 'BACK']):
        return '🛡️ DEFENSAS'

    # 4. Red de seguridad
    else:
        return '🧠 MEDIOCAMPISTAS'

# Aplicamos la función para crear la nueva columna.
df_clusters['Macro_Posicion'] = df_clusters['position_detail'].apply(clasificar_macro_posicion)


print("="*85)
print(f" 🏆 REPORTE DESCRIPTIVO: RADIOGRAFÍA DE LOS {total_clusters} ARQUETIPOS TÁCTICOS")
print("="*85)

# 1. DEFINICIÓN DE LAS DIMENSIONES TÁCTICAS (🚨 CORREGIDO CON SUFIJOS _z)
dimensiones_tacticas = {
    '⚽ ATAQUE / FINALIZACIÓN': ['xG P90_z', 'xG por Tiro_z'],
    '🧠 CREACIÓN / DISTRIBUCIÓN': ['Asistencias a Tiro P90_z', 'Pases Último Tercio P90_z', 'Pases Progresivos P90_z', '% Pases Seguridad_z', '% Centralidad Equipo_z'],
    '🛡️ DEFENSA / DESTRUCCIÓN': ['Tackles Ganados PAdj P90_z', 'Intercepciones PAdj P90_z', 'Recup. Último Tercio P90_z'],
    '🏃 MOVILIDAD / RETENCIÓN': ['Conducciones Progresivas P90_z', 'Pérdidas de Balón P90_z', '% Pases Bajo Presión_z', 'Faltas Recibidas P90_z']
}

columnas_metricas = [col for dims in dimensiones_tacticas.values() for col in dims]

# 2. CÁLCULO DE RENDIMIENTO E IMPACTO
# Score base de cada jugador (promedio de todas sus estadísticas ESTANDARIZADAS)
df_clusters['Score_Jugador'] = df_clusters[columnas_metricas].mean(axis=1)

# Agrupamos por clúster y promediamos sus variables _z
perfiles_descriptivos = df_clusters.groupby('Arquetipo_SOM')[columnas_metricas].mean()

# Calculamos el Impacto Global del clúster
perfiles_descriptivos['Impacto_Total_Cluster'] = perfiles_descriptivos.mean(axis=1)

# Calculamos el ranking de cada clúster para cada métrica individual
rankings_metricas = perfiles_descriptivos[columnas_metricas].rank(method='min', ascending=False)

# Ordenamos los clústeres del más influyente al menos influyente
clusters_ordenados = perfiles_descriptivos.sort_values(by='Impacto_Total_Cluster', ascending=False).index

# 3. BUCLE DINÁMICO DE IMPRESIÓN
for ranking_pos, cluster_id in enumerate(clusters_ordenados, start=1):

    # Extraemos información vital del clúster actual
    jugadores_del_grupo = df_clusters[df_clusters['Arquetipo_SOM'] == cluster_id]
    cantidad_jugadores = len(jugadores_del_grupo)

    medias_cluster = perfiles_descriptivos.loc[cluster_id]
    score_impacto_cluster = medias_cluster['Impacto_Total_Cluster']
    percentil_global = norm.cdf(score_impacto_cluster) * 100

    print(f"\n==================== #{ranking_pos} | ARQUETIPO TÁCTICO {cluster_id} ({cantidad_jugadores} jugadores) ====================")
    print(f"🌟 IMPACTO GLOBAL DEL GRUPO: {score_impacto_cluster:+.2f} Z-Score (Percentil Promedio: {percentil_global:.0f})")

    # --- CÁLCULO DE SCORES MACRO-DIMENSIONALES ---
    scores_dimensiones = {}
    for dim_nombre, vars_en_dim in dimensiones_tacticas.items():
        score_promedio = medias_cluster[vars_en_dim].mean()
        scores_dimensiones[dim_nombre] = score_promedio

    print("\n📊 PERFIL DEL CLÚSTER (Fuerzas Tácticas):")
    resumen_macro = " | ".join([f"{dim.split(' ')[1]} {norm.cdf(score)*100:.0f}%" for dim, score in scores_dimensiones.items()])
    print(f"   {resumen_macro}")

    # --- DESGLOSE DE VARIABLES CON RANKING TOP 3 ---
    print("\n   🔍 Desglose de Variables (Promedio del Grupo):")
    for dim_nombre, vars_en_dim in dimensiones_tacticas.items():
        print(f"      {dim_nombre}:")
        valores_dim = medias_cluster[vars_en_dim].sort_values(ascending=False)
        for var_name, valor_z in valores_dim.items():

            # Calculamos el percentil para determinar el color
            percentil = norm.cdf(valor_z) * 100

            # Escala de 5 colores según percentil
            if percentil < 20:
                indicador = "🔴" # Deficiente
            elif percentil < 40:
                indicador = "🟠" # Bajo la media
            elif percentil < 60:
                indicador = "🟡" # Promedio
            elif percentil < 80:
                indicador = "🟢" # Destacado
            else:
                indicador = "🔵" # Élite

            # Extraemos la posición en el ranking general para esta métrica
            posicion_ranking = rankings_metricas.loc[cluster_id, var_name]
            etiqueta_top = f"  [🏆 #{int(posicion_ranking)}]" if posicion_ranking <= 3 else ""

            # 🚨 NUEVO: Limpiamos el texto sacando el "_z" para la impresión en consola
            nombre_limpio = var_name.replace('_z', '')
            print(f"         {indicador} {nombre_limpio:<30} : {valor_z:+.2f} Z (Pc. {percentil:02.0f}){etiqueta_top}")

    # --- JUGADORES ÉLITE POR MACRO-POSICIÓN ---
    print("\n   ⭐ Jugadores Destacados por Línea de Juego:")

    orden_posiciones = ['🛡️ DEFENSAS', '🧠 MEDIOCAMPISTAS', '🎯 DELANTEROS']

    for macro_pos in orden_posiciones:
        # Filtramos los jugadores que pertenecen a esta macro-posición dentro del clúster
        jugadores_pos = jugadores_del_grupo[jugadores_del_grupo['Macro_Posicion'] == macro_pos]

        # Si hay jugadores de esta posición en este clúster
        if not jugadores_pos.empty:

            # Contamos cuántos hay de cada posición original exacta y lo unimos en un texto
            conteo_posiciones = jugadores_pos['position_detail'].value_counts()
            desglose_texto = " | ".join([f"{pos}: {count}" for pos, count in conteo_posiciones.items()])

            print(f"\n      {macro_pos} (Total: {len(jugadores_pos)}):")
            print(f"         📋 Desglose exacto: {desglose_texto}")

            # Extraemos e imprimimos el Top 5
            top_5_pos = jugadores_pos.sort_values(by='Score_Jugador', ascending=False).head(5)

            for index, row in top_5_pos.iterrows():
                nombre = row['player']
                equipo = row['team']
                pos_original = row['position_detail']
                score_individual = row['Score_Jugador']
                percentil_ind = norm.cdf(score_individual) * 100

                # Aplicamos la lógica visual a los jugadores
                if percentil_ind < 20:
                    ind_jugador = "🔴"
                elif percentil_ind < 40:
                    ind_jugador = "🟠"
                elif percentil_ind < 60:
                    ind_jugador = "🟡"
                elif percentil_ind < 80:
                    ind_jugador = "🟢"
                else:
                    ind_jugador = "🔵"

                print(f"         {ind_jugador} {nombre:<25} ({equipo:<12} | {pos_original:<6}) | Impacto Base: {score_individual:+.2f} Z (Pc. {percentil_ind:.0f})")
        else:
            # Insight táctico: El clúster no tiene jugadores de esta línea
            print(f"\n      {macro_pos}: [No hay jugadores de esta línea en este arquetipo]")

    print("-" * 85)

 🏆 REPORTE DESCRIPTIVO: RADIOGRAFÍA DE LOS 23 ARQUETIPOS TÁCTICOS

==================== #1 | ARQUETIPO TÁCTICO 18 (65 jugadores) ====================
🌟 IMPACTO GLOBAL DEL GRUPO: +0.96 Z-Score (Percentil Promedio: 83)

📊 PERFIL DEL CLÚSTER (Fuerzas Tácticas):
   ATAQUE 81% | CREACIÓN 86% | DEFENSA 74% | MOVILIDAD 86%

   🔍 Desglose de Variables (Promedio del Grupo):
      ⚽ ATAQUE / FINALIZACIÓN:
         🔵 xG P90                         : +1.34 Z (Pc. 91)  [🏆 #3]
         🟢 xG por Tiro                    : +0.39 Z (Pc. 65)
      🧠 CREACIÓN / DISTRIBUCIÓN:
         🔵 Asistencias a Tiro P90         : +1.81 Z (Pc. 96)  [🏆 #1]
         🔵 Pases Último Tercio P90        : +1.76 Z (Pc. 96)  [🏆 #1]
         🔵 % Centralidad Equipo           : +0.95 Z (Pc. 83)  [🏆 #2]
         🟢 % Pases Seguridad              : +0.56 Z (Pc. 71)
         🟢 Pases Progresivos P90          : +0.27 Z (Pc. 61)
      🛡️ DEFENSA / DESTRUCCIÓN:
         🔵 Recup. Último Tercio P90       : +1.72 Z (Pc. 96)  [🏆 #1]
        

## 3. Scouting de Élite: Buscando Gemelos Tácticos del "The Best" 15/16

En este bloque, ponemos a prueba el verdadero valor de nuestra arquitectura de datos aplicando un enfoque *Moneyball*. Tomaremos como referencia a los jugadores del XI Ideal de la FIFA de la temporada 2015/16 (incorporando a Gianluigi Buffon) para buscar a sus "clones" tácticos dentro del mercado.

**¿Cómo funciona este motor de búsqueda?**
Para asegurar que los jugadores recomendados no solo tengan estadísticas similares por casualidad, sino que compartan la misma esencia táctica, el algoritmo opera en tres fases:

1. **Aislamiento Topológico (Filtro SOM):** Primero, el modelo localiza la neurona exacta (Arquetipo SOM) habitada por la superestrella. La búsqueda de gemelos se restringe *exclusivamente* a los habitantes de ese mismo barrio geométrico.
2. **Similitud Geométrica (Distancia Euclidiana):** Dentro de esa neurona, el algoritmo calcula la distancia matemática exacta utilizando los 6 Componentes Principales (PCA) estandarizados, seleccionando a los 4 jugadores más cercanos en el hiperespacio.
3. **Contraste Financiero y Percentiles:** Finalmente, cruzamos las métricas brutas y calculamos el percentil de rendimiento global de cada jugador, emparejando esta información con sus valores de mercado y salarios.

**El objetivo final:** Demostrar que el modelo es capaz de detectar ineficiencias de mercado, encontrando jugadores de bajo perfil y bajo costo que logran un impacto táctico estadísticamente idéntico al de las superestrellas de clase mundial.

In [20]:
import pandas as pd
import numpy as np
from scipy.spatial.distance import cdist
from scipy.stats import percentileofscore
from IPython.display import display

print("="*85)
print(" 🏆 SCOUTING DE ÉLITE (VÍA SOM): BUSCANDO GEMELOS DEL 'THE BEST' 15/16")
print("="*85)

# 1. Definir el XI Ideal
xi_ideal = [
    'Gianluigi Buffon',
    'Daniel Alves', 'Gerard Piqué', 'Sergio Ramos', 'Marcelo Vieira',
    'Luka Modrić', 'Toni Kroos', 'Andrés Iniesta',
    'Lionel Andrés Messi', 'Luis Alberto Suárez', 'Cristiano Ronaldo'
]

# 2. Definir las columnas PCA y las Métricas Brutas a mostrar
columnas_pca = ['PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6']

metricas_brutas = [
    'xG P90', 'xG por Tiro', 'Asistencias a Tiro P90', 'Pases Progresivos P90',
    'Pases Último Tercio P90', '% Pases Seguridad', '% Pases Bajo Presión',
    '% Centralidad Equipo', 'Conducciones Progresivas P90', 'Pérdidas de Balón P90',
    'Faltas Recibidas P90', 'Tackles Ganados PAdj P90', 'Intercepciones PAdj P90',
    'Recup. Último Tercio P90', 'Influencia Global P90'
]

df_busqueda = df_clusters.copy()

# 🚨 PARCHE PREVENTIVO: Asegurarnos de que las columnas PCA sean estrictamente números
for col in columnas_pca:
    df_busqueda[col] = pd.to_numeric(df_busqueda[col], errors='coerce')

for estrella in xi_ideal:
    # Buscar a la estrella
    jugador_target = df_busqueda[df_busqueda['player'].str.contains(estrella, case=False, na=False)]

    if jugador_target.empty:
        print(f"⚠️ {estrella} no fue encontrado en la base de datos.")
        print("-" * 85)
        continue

    # Extraer datos de la estrella
    indice_estrella = jugador_target.index[0]
    nombre_real = df_busqueda.loc[indice_estrella, 'player']
    neurona_estrella = df_busqueda.loc[indice_estrella, 'Arquetipo_SOM']

    # EL FILTRO SOM: Aislamos SOLO a los jugadores de su misma neurona
    df_misma_neurona = df_busqueda[df_busqueda['Arquetipo_SOM'] == neurona_estrella].copy()

    # Verificar que haya más jugadores en esa neurona
    if len(df_misma_neurona) <= 1:
        print(f"⚠️ {nombre_real} está solo en la Neurona {neurona_estrella}. No hay gemelos posibles en este arquetipo.")
        print("-" * 85)
        continue

    # 🚨 FIX: Extraer coordenadas PCA y OBLIGAR a que sean formato float (números decimales)
    coordenadas_estrella = df_misma_neurona.loc[indice_estrella, columnas_pca].values.astype(float).reshape(1, -1)
    coordenadas_neurona = df_misma_neurona[columnas_pca].values.astype(float)

    # Calcular distancia euclidiana intra-neurona
    distancias = cdist(coordenadas_estrella, coordenadas_neurona, metric='euclidean')[0]
    df_misma_neurona['Dist_Euclidiana_Intra_Neurona'] = distancias

    # Seleccionar a los 2 más cercanos dentro de la neurona
    gemelos = df_misma_neurona[df_misma_neurona['player'] != nombre_real].sort_values(by='Dist_Euclidiana_Intra_Neurona').head(4)

    # Preparar tabla comparativa
    df_comparativa = pd.concat([df_misma_neurona.loc[[indice_estrella]], gemelos])

    tabla_final = pd.DataFrame()
    tabla_final['Jugador'] = df_comparativa['player']
    tabla_final['Equipo'] = df_comparativa['team']
    tabla_final['posicion'] = df_comparativa['position_detail']



    # Manejo seguro de la edad
    tabla_final['Edad'] = pd.to_numeric(df_comparativa['age'], errors='coerce').fillna(0).astype(int)
    tabla_final['Neurona SOM'] = df_comparativa['Arquetipo_SOM']

    # Precios y Salarios (con conversión a numérico por seguridad)
    tabla_final['Valor de Mercado'] = pd.to_numeric(df_comparativa['value_eur'], errors='coerce').apply(
        lambda x: f"€ {x:,.0f}" if pd.notnull(x) else "Desconocido"
    )
    tabla_final['Salario Semanal'] = pd.to_numeric(df_comparativa['wage_eur'], errors='coerce').apply(
        lambda x: f"€ {x:,.0f}" if pd.notnull(x) else "Desconocido"
    )

    # Calcular percentiles globales (usando la base completa, no solo la neurona)
    for metrica in metricas_brutas:
        if metrica in df_busqueda.columns:
            valores_formateados = []
            for idx in df_comparativa.index:
                # Nos aseguramos de que el valor bruto sea numérico
                valor_bruto = float(df_busqueda.loc[idx, metrica])
                serie_metrica = pd.to_numeric(df_busqueda[metrica], errors='coerce').dropna()
                percentil = percentileofscore(serie_metrica, valor_bruto)
                valores_formateados.append(f"{valor_bruto:.2f} (Pc. {percentil:02.0f})")

            tabla_final[metrica] = valores_formateados

    print(f"\n🌟 OBJETIVO: {nombre_real.upper()} (Neurona SOM: {neurona_estrella})")
    display(tabla_final.set_index('Jugador').T)
    print("-" * 85)

print("✅ Scouting intracelular completado.")

 🏆 SCOUTING DE ÉLITE (VÍA SOM): BUSCANDO GEMELOS DEL 'THE BEST' 15/16

🌟 OBJETIVO: GIANLUIGI BUFFON (Neurona SOM: 5)


Jugador,Gianluigi Buffon,Kevin Trapp,Claudio Andrés Bravo Muñoz,Eugenio Lamanna,Samir Handanovič
Equipo,Juventus,Paris Saint-Germain,Barcelona,Genoa,Inter Milan
posicion,Goalkeeper,Goalkeeper,Goalkeeper,Goalkeeper,Goalkeeper
Edad,37,24,32,25,30
Neurona SOM,5,5,5,5,5
Valor de Mercado,"€ 9,000,000","€ 21,500,000","€ 17,500,000","€ 1,800,000","€ 19,000,000"
Salario Semanal,"€ 130,000","€ 120,000","€ 130,000","€ 25,000","€ 120,000"
xG P90,0.00 (Pc. 04),0.00 (Pc. 04),0.00 (Pc. 04),0.00 (Pc. 04),0.00 (Pc. 04)
xG por Tiro,0.00 (Pc. 04),0.00 (Pc. 04),0.00 (Pc. 04),0.00 (Pc. 04),0.00 (Pc. 04)
Asistencias a Tiro P90,0.00 (Pc. 03),0.00 (Pc. 03),0.00 (Pc. 03),0.08 (Pc. 14),0.00 (Pc. 03)
Pases Progresivos P90,17.13 (Pc. 70),18.27 (Pc. 76),18.67 (Pc. 77),21.99 (Pc. 88),21.07 (Pc. 85)


-------------------------------------------------------------------------------------

🌟 OBJETIVO: DANIEL ALVES DA SILVA (Neurona SOM: 3)


Jugador,Daniel Alves da Silva,Claudio Marchisio,Yuri Berchiche Izeta,Issiaga Sylla,Milan Badelj
Equipo,Barcelona,Juventus,Real Sociedad,Gazélec Ajaccio,Fiorentina
posicion,Right Back,Center Defensive Midfield,Left Back,Left Back,Right Defensive Midfield
Edad,32,29,25,21,26
Neurona SOM,3,3,3,3,3
Valor de Mercado,"€ 16,000,000","€ 26,500,000","€ 1,800,000","€ 1,100,000","€ 6,500,000"
Salario Semanal,"€ 170,000","€ 150,000","€ 20,000","€ 8,000","€ 80,000"
xG P90,0.02 (Pc. 32),0.03 (Pc. 39),0.02 (Pc. 26),0.02 (Pc. 32),0.04 (Pc. 43)
xG por Tiro,0.04 (Pc. 24),0.03 (Pc. 15),0.03 (Pc. 15),0.04 (Pc. 20),0.04 (Pc. 19)
Asistencias a Tiro P90,0.66 (Pc. 67),0.91 (Pc. 81),0.62 (Pc. 65),0.74 (Pc. 73),1.28 (Pc. 91)
Pases Progresivos P90,17.71 (Pc. 73),23.32 (Pc. 91),21.48 (Pc. 86),19.04 (Pc. 78),24.06 (Pc. 92)


-------------------------------------------------------------------------------------

🌟 OBJETIVO: GERARD PIQUÉ BERNABÉU (Neurona SOM: 1)


...

-------------------------------------------------------------------------------------

🌟 OBJETIVO: LUIS ALBERTO SUÁREZ DÍAZ (Neurona SOM: 2)


Jugador,Luis Alberto Suárez Díaz,Lucas Pérez Martínez,Cristiano Ronaldo dos Santos Aveiro,Odion Jude Ighalo,Bojan Krkíc Pérez
Equipo,Barcelona,RC Deportivo La Coruña,Real Madrid,Watford,Stoke City
posicion,Center Forward,Center Forward,Left Wing,Left Center Forward,Center Attacking Midfield
Edad,28,26,30,26,24
Neurona SOM,2,2,2,2,2
Valor de Mercado,"€ 69,000,000","€ 10,000,000","€ 85,500,000","€ 2,700,000","€ 21,500,000"
Salario Semanal,"€ 300,000","€ 60,000","€ 475,000","€ 25,000","€ 70,000"
xG P90,0.78 (Pc. 100),0.43 (Pc. 98),0.89 (Pc. 100),0.46 (Pc. 99),0.21 (Pc. 90)
xG por Tiro,0.20 (Pc. 98),0.16 (Pc. 92),0.14 (Pc. 87),0.15 (Pc. 90),0.13 (Pc. 84)
Asistencias a Tiro P90,1.28 (Pc. 91),1.66 (Pc. 97),0.95 (Pc. 83),0.84 (Pc. 79),1.15 (Pc. 89)
Pases Progresivos P90,5.08 (Pc. 25),4.60 (Pc. 23),4.28 (Pc. 21),2.15 (Pc. 08),7.85 (Pc. 35)


-------------------------------------------------------------------------------------

🌟 OBJETIVO: CRISTIANO RONALDO DOS SANTOS AVEIRO (Neurona SOM: 2)


Jugador,Cristiano Ronaldo dos Santos Aveiro,Karim Benzema,Gonzalo Gerardo Higuaín,Bojan Krkíc Pérez,Lucas Pérez Martínez
Equipo,Real Madrid,Real Madrid,Napoli,Stoke City,RC Deportivo La Coruña
posicion,Left Wing,Center Forward,Center Forward,Center Attacking Midfield,Center Forward
Edad,30,27,27,24,26
Neurona SOM,2,2,2,2,2
Valor de Mercado,"€ 85,500,000","€ 46,500,000","€ 34,500,000","€ 21,500,000","€ 10,000,000"
Salario Semanal,"€ 475,000","€ 230,000","€ 190,000","€ 70,000","€ 60,000"
xG P90,0.89 (Pc. 100),0.76 (Pc. 100),0.75 (Pc. 100),0.21 (Pc. 90),0.43 (Pc. 98)
xG por Tiro,0.14 (Pc. 87),0.18 (Pc. 95),0.14 (Pc. 86),0.13 (Pc. 84),0.16 (Pc. 92)
Asistencias a Tiro P90,0.95 (Pc. 83),1.39 (Pc. 94),1.40 (Pc. 94),1.15 (Pc. 89),1.66 (Pc. 97)
Pases Progresivos P90,4.28 (Pc. 21),5.57 (Pc. 27),5.05 (Pc. 25),7.85 (Pc. 35),4.60 (Pc. 23)


-------------------------------------------------------------------------------------
✅ Scouting intracelular completado.


# 🏁 Conclusión y Trabajo Futuro: De la Descripción a la Predicción

## 🏆 El Hito Actual: Scouting Cuantitativo y "Moneyball" Táctico
Con la ejecución de este motor de búsqueda, hemos culminado la etapa descriptiva y diagnóstica de nuestro pipeline analítico. Al utilizar el equipo **FIFA The Best de la temporada 2015/16** como referencia, demostramos que la arquitectura de **PCA + Mapas Autoorganizados (SOM)** es capaz de leer el fútbol más allá de las posiciones nominales.

El modelo logró agrupar a los jugadores en "Arquetipos Tácticos" puros, permitiéndonos identificar asimetrías de mercado fascinantes: gemelos estadísticos que operaban en los mismos percentiles de rendimiento que las superestrellas mundiales, pero a una fracción de su valor de mercado (ej. perfiles como Lucas Pérez o Riyad Mahrez operando topológicamente cerca de Cristiano Ronaldo y Lionel Messi).

---

## 🚀 Trabajo Futuro: Backtesting y Validación Temporal (2016 - 2017)

El ecosistema de datos actual nos indica **quiénes rindieron como estrellas en el pasado**, pero el verdadero Santo Grial de la analítica deportiva es evaluar la sostenibilidad de ese rendimiento. Para la próxima iteración de este proyecto, el objetivo será transicionar hacia la **Analítica Predictiva y Validación de Mercado**, bajo la siguiente hoja de ruta:

### 1. Validación de Fichajes (El Veredicto del Mercado)
Se extraerá una lista de "Gemas Ocultas" (jugadores subvalorados con alto impacto táctico intra-neurona) detectadas por nuestro SOM al cierre de la temporada 15/16. Esta lista se cruzará con la base de datos de transferencias del verano de 2016 para responder una pregunta clave: *¿Ficharon los clubes de élite a los jugadores que nuestro algoritmo recomendaba en silencio?*

### 2. Backtesting de Rendimiento (Temporada 16/17)
Se integrará el dataset de eventos de la temporada 2016/2017 para evaluar la progresión de los "gemelos tácticos" descubiertos. El objetivo será medir:
* **Estabilidad del Arquetipo:** ¿Mantuvo el jugador su posición topológica en el mapa tras un cambio de club o de sistema táctico?
* **Proyección de Métricas:** ¿Qué tan sostenible es un salto repentino al percentil 90+ en métricas de finalización o creación?

### 3. Refinamiento Estadístico (Z-Scores Contextuales)
Para corregir el sesgo inherente de comparar perfiles defensivos contra ofensivos a nivel global, los futuros *scores* de impacto se calcularán de manera aislada. Se estandarizarán las métricas (Z-Score) exclusivamente dentro de cada Macro-Posición o intra-Arquetipo, garantizando que cada jugador sea evaluado estrictamente en su propio oficio y contexto.

> *El modelo actual ha demostrado que puede entender y decodificar el talento táctico. El próximo paso será demostrar que también puede anticiparlo.*